In [5]:
r2_scores={}

In [3]:
#Importing necessarry libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import os
import polars as pl
import pandas as pd
from torch.utils.data import Dataset
import torch.optim as optim
import gc
import wandb
import json
from torch.utils.data import Subset
from collections import defaultdict



In [7]:
class EmbeddedDataset(Dataset):
    def __init__(self, data):
        self.data= data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx,:,4:83], self.data[idx,-1,83],self.data[idx,-1,3],self.data[idx,:,2].to(int)

class EmbeddedMLPDataset(Dataset):
    def __init__(self, data):
        self.data= data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx,4:83], self.data[idx,83],self.data[idx,3],self.data[idx,2].to(int)


class FullDataset(Dataset):
    def __init__(self, data):
        self.data= data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx,:,4:83], self.data[idx,-1,83],self.data[idx,-1,3],self.data[idx,:,2].to(int)

class FullDatasetMLP(Dataset):
    def __init__(self, data):
        self.data= data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx,4:83], self.data[idx,83],self.data[idx,3]

In [ ]:
# Different Types of losses

def w_mse(predictions: torch.Tensor, targets: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    """
    Compute the weighted Mean Squared Error (MSE) loss.

    Args:
        predictions (torch.Tensor): The predicted values from the model.
        targets (torch.Tensor): The true target values.
        weights (torch.Tensor): The weights for each sample (should have the same shape as predictions and targets).

    Returns:
        torch.Tensor: The weighted MSE loss.
    """
    # Define the MSE loss function with no reduction
    loss_fn = nn.MSELoss(reduction='none')

    # Compute the MSE loss (element-wise)
    mse_loss = loss_fn(predictions, targets)

    # Multiply the loss by the weights
    weighted_loss = mse_loss * weights

    # Return the mean of the weighted loss
    return weighted_loss.mean()


def w_mae(targets, predictions, weights):
    """
    Calculate the Weighted Mean Absolute Error (WMAE).

    Parameters:
    - targets (array-like): Ground truth values.
    - predictions (array-like): Predicted values.
    - weights (array-like): Weights for each data point.

    Returns:
    - float: The weighted mean absolute error.
    """
    # Convert inputs to NumPy arrays for consistency


    # Validate inputs
    if not (len(targets) == len(predictions) == len(weights)):
        raise ValueError("Targets, predictions, and weights must have the same length.")

    # Calculate the weighted absolute errors
    absolute_errors = torch.abs(targets - predictions)
    weighted_errors = absolute_errors * weights

    # Compute the weighted mean absolute error
    wmae = torch.sum(weighted_errors) / torch.sum(weights)
    return wmae


def h_loss(targets, predictions, delta=1.0):
    """
    Calculate the Huber Loss.

    Parameters:
    - targets (array-like): Ground truth values.
    - predictions (array-like): Predicted values.
    - delta (float): The threshold where the loss transitions from quadratic to linear.

    Returns:
    - float: The Huber loss.
    """
    # Convert inputs to NumPy arrays for consistency


    # Validate inputs
    if len(targets) != len(predictions):
        raise ValueError("Targets and predictions must have the same length.")

    # Calculate the absolute differences
    errors = torch.abs(targets - predictions)

    # Compute Huber Loss
    is_small_error = errors <= delta
    small_error_loss = 0.5 * (errors[is_small_error] ** 2)
    large_error_loss = delta * (errors[~is_small_error] - 0.5 * delta)

    huber = torch.sum(small_error_loss) + torch.sum(large_error_loss)
    return huber / len(targets)

def w_h_loss(targets, predictions, weights, delta=1.0):
    """
    Calculate the Weighted Huber Loss.

    Parameters:
    - targets (array-like): Ground truth values.
    - predictions (array-like): Predicted values.
    - weights (array-like): Weights for each data point.
    - delta (float): The threshold where the loss transitions from quadratic to linear.

    Returns:
    - float: The weighted Huber loss.
    """
    # Convert inputs to NumPy arrays for consistency


    # Validate inputs
    if not (len(targets) == len(predictions) == len(weights)):
        raise ValueError("Targets, predictions, and weights must have the same length.")

    # Calculate the absolute differences
    errors = torch.abs(targets - predictions)

    # Compute Huber Loss
    is_small_error = errors <= delta
    small_error_loss = 0.5 * (errors[is_small_error] ** 2) * weights[is_small_error]
    large_error_loss = delta * (errors[~is_small_error] - 0.5 * delta) * weights[~is_small_error]

    huber = torch.sum(small_error_loss) + np.sum(large_error_loss)
    return huber / torch.sum(weights)


def mae(targets, predictions):
    """
    Calculate the Mean Absolute Error (MAE).

    Parameters:
    - targets (array-like): Ground truth values.
    - predictions (array-like): Predicted values.

    Returns:
    - float: The mean absolute error.
    """
    # Convert inputs to NumPy arrays for consistency


    # Validate inputs
    if len(targets) != len(predictions):
        raise ValueError("Targets and predictions must have the same length.")

    # Calculate absolute differences
    absolute_errors = torch.abs(targets - predictions)

    # Compute the mean absolute error
    mae = torch.mean(absolute_errors)
    return mae

In [ ]:
# Helper functions

def weighted_r2(predictions, targets, weights):
    """
    Compute the weighted R^2 score using PyTorch tensors.

    Parameters:
        predictions (tensor): Predicted values.
        targets (tensor): Actual target values.
        weights (tensor): Weights corresponding to each data point.

    Returns:
        tensor: Weighted R^2 score.
    """
    # Ensure weights are positive
    assert torch.all(weights >= 0), "Weights must be non-negative."

    # Compute weighted mean of targets
    weighted_mean = torch.sum(weights * targets) / torch.sum(weights)

    # Compute the weighted sum of squares
    ss_total = torch.sum(weights * (targets - weighted_mean) ** 2)
    ss_residual = torch.sum(weights * (targets - predictions) ** 2)

    # Handle case when total variance is zero
    if ss_total.item() == 0:
        return torch.tensor(1.0) if ss_residual.item() == 0 else torch.tensor(0.0)

    # Compute weighted R^2
    r2 = 1 - (ss_residual / ss_total)
    return r2

def expert_data(x):
    # expects polars data first col = dateid 2nd col = time id and 3rd col = symbol id
    x = x.sort("symbol_id")
    cols_to_remove = [
    'responder_0',
    'responder_1',
    'responder_2',
    'responder_3',
    'responder_4',
    'responder_5',
    'responder_7',
    'responder_8'
    ]

    # Drop the specified columns
    x = x.drop(cols_to_remove)


    # Drop the 'weight' column from the original DataFrame
    x = x.fill_null(0)
    x = x.to_numpy()

    x = torch.tensor(x)

    x_symbols = seperate_symbols(x)

    return x_symbols




# seprating data into each batch for symbols
def seperate_symbols(data):
    """
    Groups rows into batches until a new (symbol_id)  occurs.
    Assumes the data is sorted by (symbol id).

    Args:
       torch tensor 3 column should be symbol id

    Returns:
        List[torch.Tensor]: Batches of rows grouped by the same (symbol_id).
    """
    batches = []
    current_batch = []
    current_symbol = None

    for row in data:
        # Extract (date, time) from the row

        symbol = int(row[2])

        if current_symbol is None or symbol == current_symbol:
            # Add to the current batch
            current_batch.append(row)
            current_symbol = symbol
        else:
            # New combination: finalize the current batch and start a new one
            batches.append(torch.stack(current_batch))
            current_batch = [row]
            current_symbol = symbol

    # Add the last batch if any
    if current_batch:
        batches.append(torch.stack(current_batch))

    return batches



# seperate based on date time
def batch_symbols(data):
    # expects tensor data sorted by date id and time id
    """
    Groups rows into batches until a new (date id time id)  occurs.
    Assumes the data is sorted by (date id time id).

    Args:
       torch tensor 1,2 column should be date id time id

    Returns:
        List[torch.Tensor]: Batches of rows grouped by the same (symbol_id).
    """
    batches = []
    current_batch = []
    current_combination = None

    for row in data:
        # Extract (date, time) from the row

        combination = tuple([int(row[0].item()),int(row[1].item())])

        if current_combination is None or combination  == current_combination:
            # Add to the current batch
            current_batch.append(row)
            current_combination = combination
        else:
            # New combination: finalize the current batch and start a new one
            batches.append(torch.stack(current_batch))
            current_batch = [row]
            current_combination = combination

    # Add the last batch if any
    if current_batch:
        batches.append(torch.stack(current_batch))

    return batches



#create dataset without sequences
# expects a polars dataframe

def prepare_data(data):
    cols_to_remove = [
    'responder_0',
    'responder_1',
    'responder_2',
    'responder_3',
    'responder_4',
    'responder_5',
    'responder_7',
    'responder_8'
    ]

    # Drop the specified columns
    data = data.drop(cols_to_remove)


    data = data.fill_null(0)
    data = data.to_numpy()





    return data
    # returns numpy so change to tensor and may need to unsqueeze

def eval_prepare(data):
    data = data.sort("date_id","time_id")
    cols_to_remove = [
    'responder_0',
    'responder_1',
    'responder_2',
    'responder_3',
    'responder_4',
    'responder_5',
    'responder_7',
    'responder_8'
    ]

    # Drop the specified columns
    data = data.drop(cols_to_remove)


    # Drop the 'weight' column from the original DataFrame
    data = data.fill_null(0)
    data = data.to_numpy()

    return data

In [ ]:
def prepare_sequences(data,number_of_seqs):
    data = data.sort("symbol_id","date_id","time_id")
    cols_to_remove = [
    'responder_0',
    'responder_1',
    'responder_2',
    'responder_3',
    'responder_4',
    'responder_5',
    'responder_7',
    'responder_8'
    ]

    # Drop the specified columns
    data = data.drop(cols_to_remove)


    # Drop the 'weight' column from the original DataFrame
    data = data.fill_null(0)
    data = data.to_numpy()
    data = torch.tensor(data)
    data[:,4:83]= normalize(data[:,4:83])
    sequences = create_sequences(data,number_of_seqs)




    return sequences


def create_sequences(data, sequence_len):
    sequence_length = sequence_len

    class_sequences = {}


    for row in data:
        class_label = int(row[0])
        if class_label not in class_sequences:
            class_sequences[class_label] = []
        class_sequences[class_label].append(row)

    # Now, create sequences for each class
    sequences = []

    for class_label, rows in class_sequences.items():
        # Convert the list of rows into a numpy array
        class_data = np.array(rows)

        # Create sequences from the class data
        for i in range(len(class_data) - sequence_length + 1):
            sequences.append(class_data[i:i + sequence_length])

    # Convert sequences to a numpy array and then to a PyTorch tensor
    sequences = np.array(sequences)
    tensor_sequences = torch.tensor(sequences, dtype=torch.float32)

    return tensor_sequences

In [ ]:
# Training function

def train(model, dataloader, loss_fn, optimizer, epochs=10):
    model.train()  # Set the model to training mode
    i=0
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, targets, weights in dataloader:
            inputs, targets, weights = inputs.to(device), targets.to(device), weights.to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            predictions = model(inputs)


            # Calculate loss
            loss = mae(predictions, targets)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # del inputs, targets, weights  # Delete the tensors to remove references
            # torch.cuda.empty_cache()

            running_loss += loss.item()

        # Print average loss for the epoch
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss / len(dataloader):.4f}")


def normalize(features: torch.Tensor) -> torch.Tensor:
    """
    Normalizes a tensor to have mean 0 and standard deviation 1.

    Args:
        features (torch.Tensor): The input tensor of features.

    Returns:
        torch.Tensor: The normalized tensor.
    """
    mean = features.mean(dim=0, keepdim=True)  # Calculate mean along columns
    std = features.std(dim=0, keepdim=True)  # Calculate std along columns
    std[std == 0] = 1  # Avoid division by zero for constant features
    normalized_features = (features - mean) / std
    return normalized_features


In [ ]:
#scanning all the datasets using polars
part_0 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=0/part-0.parquet")
part_1 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=1/part-0.parquet")
part_9 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=9/part-0.parquet")
part_8 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=8/part-0.parquet")
part_7 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=7/part-0.parquet")
part_6 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=6/part-0.parquet")
part_5 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=5/part-0.parquet")
part_4 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=4/part-0.parquet")
part_3 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=3/part-0.parquet")
part_2 = pl.scan_parquet("/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=2/part-0.parquet")

Training and evaluating Models without symbol ids to throw away the information in symbol ids.
This is done as the first step to form a baseline.

In [ ]:
# Models
# Define the simple MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)  # Apply ReLU activation to the hidden layer
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)   # Output layer
        return x

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()

        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)

        # We take the output from the last time step for classification
        out = self.fc(lstm_out[:, -1, :])
        return out


class EmbeddedMLP(nn.Module):
    def __init__(self, num_features, num_symbols, embedding_dim):
        super(EmbeddedMLP, self).__init__()
        # Embedding layer for symbol_id
        self.embedding = nn.Embedding(num_symbols, embedding_dim)
        # Fully connected layers
        self.fc1 = nn.Linear(num_features + embedding_dim, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, features, symbol_id):
        # Pass symbol_id through embedding layer
        embedded_symbols = self.embedding(symbol_id)
        # Concatenate features and embeddings
        x = torch.cat([features, embedded_symbols], dim=1)
        # Forward pass through fully connected layers
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

#Learned feature engineering

class FLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, input_size, hidden_size, output_size, num_layers=1,r_features_size = 64):
        super(FLSTM, self).__init__()

        # Embedding layer for symbol IDs
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # LSTM layer
        self.lstm = nn.LSTM(input_size + embedding_dim, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, r_features_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x, symbol_ids):
        # Embedding lookup for symbol IDs
        symbol_embeds = self.embedding(symbol_ids)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        # Concatenate feature vectors with symbol embeddings
        x = torch.cat((x, symbol_embeds), dim=-1)

        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)

        # We take the output from the last time step for classification
        out = self.fc(lstm_out[:, -1, :])
        return out

# LSTM with embeddings to extract symbol information
class ELSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, input_size, hidden_size, output_size, num_layers=1):
        super(ELSTM, self).__init__()

        # Embedding layer for symbol IDs
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # LSTM layer
        self.lstm = nn.LSTM(input_size + embedding_dim, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, symbol_ids):
        # Embedding lookup for symbol IDs
        symbol_embeds = self.embedding(symbol_ids)

        # Concatenate feature vectors with symbol embeddings
        x = torch.cat((x, symbol_embeds), dim=-1)

        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)

        # We take the output from the last time step for classification
        out = self.fc(lstm_out[:, -1, :])
        return out

In [ ]:
# BaseLine MLP models

In [ ]:
data = prepare_data(part_0)
x = data[:,4:83]
y = data[:,83]
w = data[:,3]
print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:
x = torch.tensor(x)
x = normalize(x)
y = torch.tensor(y)
w = torch.tensor(w)
print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dataset = TensorDataset(x,y,w)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

In [ ]:
#Sanity check
model = MLP(input_size=79, hidden_size=256, output_size=1)
for inputs, targets, weights in dataloader:
    outputs = model(inputs)
    print(outputs.shape)
    print(inputs.shape)
    print(targets.shape)
    print(weights.shape)
    break

In [ ]:
#Train The model
model = MLP(input_size=79, hidden_size=256, output_size=1)
model = model.to(device)
model = torch.nn.DataParallel(model, device_ids=[0, 1])
loss_fn = nn.MSELoss()

# Use Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
train(model, dataloader, loss_fn, optimizer, epochs=10)

In [ ]:
# Evaluation without online training
test_data = part_1.collect()
data = eval_prepare(test_data)
x = data[:,4:83]
y = data[:,83]
w = data[:,3]

print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:
# removing the symbol and data and time id

x = torch.tensor(x)
y = torch.tensor(y)
w = torch.tensor(w)
print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:
test_dataset= TensorDataset(x)
test_dataloader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
# Evaluation run
predictions=[]
model.eval()
with torch.no_grad():
    for inputs in test_dataloader:
        # Get the model outputs (predictions) for the current batch
        inputs[0].to(device)
        outputs = model(inputs[0])  # inputs[0] since DataLoader gives a tuple (input, label)

        # Append the outputs to the predictions list
        predictions.append(outputs)

# Concatenate all predictions into a single tensor
test_predictions = torch.cat(predictions, dim=0)
print(test_predictions.shape)

cpu = torch.device("cpu")
test_predictions = test_predictions.to(cpu)
w_r2 = weighted_r2(test_predictions,y,w)
print(w_r2)


In [ ]:
#Evaluation with online training


In [ ]:
# prepare data
test_data = part_1.collect()
data = eval_prepare(test_data)
print(data.shape)

In [ ]:
# prepare data
data = torch.tensor(data)
data[:,4:83]= normalize(data[:,4:83])
print(data.shape)
batches = batch_symbols(data)
print(len(batches))
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
predictions=[]
targets =[]
weights =[]
sequence_size = 5
for batch_idx, inputs in enumerate(batches[0:16000]):
        sequence = batches[batch_idx:batch_idx + sequence_size]

        features = inputs[:,4:83]
        y= inputs[:,83]
        w= inputs[:,3]


    # Feed sequence_tensor to the LSTM
        outputs = model(features)
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass

        predictions.append(outputs.squeeze())  # Store predictions

        # Compute loss
        targets.append(y.squeeze())
        weights.append(w.squeeze())

        loss = w_mse(outputs.squeeze(), y.float(),w.float())

        # Backward pass
        loss.backward()
        optimizer.step()



print(f"Stored predictions: {len(predictions)} batches")
#prepare x,y,w for wr2
y=torch.cat([tensor for tensor in targets])
w=torch.cat([tensor for tensor in weights])
test_predictions=torch.cat([tensor for tensor in predictions])
print(test_predictions.shape)
print(y.shape)
print(w.shape)

wr2 = weighted_r2(test_predictions,y,w)
wr2

In [ ]:
# MLP with symbol information

In [ ]:
data[:,4:83]= normalize(data[:,4:83])


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dataset = EmbeddedMLPDataset(data)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False)
model = EmbeddedMLP(num_features=79, num_symbols=39, embedding_dim=10)
model = model.to(device)
model = torch.nn.DataParallel(model, device_ids=[0, 1])
loss_fn = nn.MSELoss()

# Use Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
model.train()  # Set the model to training mode
for epoch in range(10):
    running_loss = 0.0
    for inputs, targets, weights,symbol in dataloader:
        inputs, targets, weights,symbol = inputs.to(device), targets.to(device), weights.to(device),symbol.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(inputs,symbol)


        # Calculate loss
        loss = mae(predictions, targets)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # del inputs, targets, weights  # Delete the tensors to remove references
        # torch.cuda.empty_cache()

        running_loss += loss.item()

    # Print average loss for the epoch
    print(f"Epoch [{epoch+1}/{10}], Loss: {running_loss / len(dataloader):.4f}")

In [ ]:
# EVALUATION LOOP IS THE SAME FOR BOTH SIMPLE EVALUATION AND EVALUATION WITH ONLINE TRAINING SO THE ABOVE EVALUATION LOOPS WERE REUSED


In [ ]:
# TRAINING A SEPERATE MODEL FOR EACH SYMBOL

In [ ]:
symbols = data[:, 2].astype(int)
# Group indices by symbol
symbol_indices = defaultdict(list)
for idx, symbol in enumerate(symbols):
    symbol_indices[symbol].append(idx)


full_dataset = FullDatasetMLP(data)

# Create subsets for each symbol
symbol_datasets = {symbol: Subset(full_dataset, indices) for symbol, indices in symbol_indices.items()}

dataloaders = {symbol: DataLoader(dataset, batch_size=256, shuffle=True) for symbol, dataset in symbol_datasets.items()}

In [ ]:
models = {}
criterion = nn.MSELoss()
for symbol, dataloader in dataloaders.items():
     model = MLP(79,256,1)
     optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
     for epoch in range(10):
        for features,targets,weights in dataloader:
            features= features[:,3:]
            targets= targets[:,3:]
            weights= weights[:,3:]
            optimizer.zero_grad()
            predictions = model(features)
            loss = criterion(predictions.squeeze(), targets.squeeze())
            loss.backward()
            optimizer.step()

    # Store trained model
     models[symbol] = model

models[0]

In [ ]:
test_data = part_1.collect()
data = eval_prepare(data)
x = data[:,4:83]
y = data[:,83]
w = data[:,3]
print(x.shape)
print(y.shape)
print(w.shape)
data[:,4:83] = normalize(data[:,4:83])
y = torch.tensor(y)
w = torch.tensor(w)
x = torch.tensor(x)
print(x.shape)
print(y.shape)
print(w.shape)
x_batches = batch_symbols(x)
y_batches= batch_symbols(y)
w_batches = batch_symbols(w)
print(len(x_batches))
print(len(y_batches))
print(len(w_batches))
loss_fn = nn.MSELoss()

In [ ]:
optimizers={}
for key,model in models.items():
    optimizers[key] = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
batch_size = 32
predictions = []
weights = []
targets = []

for batch_idx, inputs in enumerate(x_batches[0:10000]):
    batch_features = []
    batch_targets = []
    batch_weights = []
    batch_symbols = []

    for i, input in enumerate(inputs):
        features = input[3:]
        symbol = input[2].to(dtype=torch.int).item()

        # Check if symbol is in models and optimizers
        if symbol not in models or symbol not in optimizers:
            continue

        batch_features.append(features)
        batch_targets.append(y_batches[batch_idx][i][-1])
        batch_weights.append(w_batches[batch_idx][i][-1])
        batch_symbols.append(symbol)

    # Check if batch is not empty
    if not batch_features:
        continue

    # Stack batch features, targets, and weights
    batch_features = torch.stack(batch_features)
    batch_targets = torch.stack(batch_targets)
    batch_weights = torch.stack(batch_weights)

    # Zero gradients
    for symbol in set(batch_symbols):  # Use set to avoid redundant `zero_grad` calls
        optimizers[symbol].zero_grad()

    # Forward pass
    batch_outputs = []
    for symbol, features in zip(batch_symbols, batch_features):
        output = models[symbol](features)
        batch_outputs.append(output)

    # Compute loss
    batch_loss = 0
    for output, target, weight in zip(batch_outputs, batch_targets, batch_weights):
        loss = w_mse(output.squeeze(), target.float(), weight.float())
        batch_loss += loss

    # Backward pass
    batch_loss.backward()
    for symbol in set(batch_symbols):
        optimizers[symbol].step()

    # Store predictions, targets, and weights
    predictions.extend(batch_outputs)
    targets.extend(batch_targets)
    weights.extend(batch_weights)

print(f"Stored predictions: {len(predictions)}")
print(f"Stored targets: {len(targets)}")
print(f"Stored weights: {len(weights)}")

In [ ]:
predictions = [el.detach() for el in predictions]
weights = [el.detach() for el in weights]
targets = [el.detach() for el in targets]
predictions = torch.tensor(predictions)
weights =  torch.tensor(weights)
targets = torch.tensor(targets)
print(predictions.shape)
print(weights.shape)
print(targets.shape)
wr2 = weighted_r2(predictions,targets,weights)
print(wr2)

In [ ]:
# LSTMs Models

In [ ]:
#loading the necessary datasets
data = part_0.collect()

In [ ]:
sequences = prepare_sequences(data,5)

print(sequences.shape)


In [ ]:
x = sequences[:,:,4:]
w = sequences[:,:,3]
y = sequences[:,:,83]
print(x.shape)
print(y.shape)
print(w.shape)


In [ ]:
# sequence to vector modeling
w= w[:,-1,:]
y = y[:,-1,:]
print(y.shape)
print(w.shape)

In [ ]:
# Create a dataset and dataloader
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dataset = TensorDataset(x,y,w)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

In [ ]:

#Sanity check
model = LSTMModel(input_size=79, hidden_size=256, output_size=1)
for inputs, targets, weights in dataloader:
    outputs = model(inputs)
    print(outputs.shape)
    print(inputs.shape)
    print(targets.shape)
    print(weights.shape)
    break

In [ ]:
#Train The model
model = LSTMModel(input_size=79, hidden_size=256, output_size=1)
model = model.to(device)
model = torch.nn.DataParallel(model, device_ids=[0, 1])
loss_fn = nn.MSELoss()

# Use Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
train(model, dataloader, loss_fn, optimizer, epochs=10)

In [ ]:
#memory management
del data
gc.collect()

In [ ]:
#Evaluation

In [ ]:
# loading data for evaluations
test_data = part_1.collect()
x,y,w = prepare_sequences(test_data,5)
print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:
# removing the symbol and data and time id
x = x[:,:,1:]
y = y[:,:,1:]
w = w[:,:,1:]
print(x.shape)
print(y.shape)
print(w.shape)


In [ ]:
#seq to vec
y = y[:,-1,:]
w = w[:,-1,:]
print(x.shape)
print(y.shape)
print(w.shape)

In [ ]:

# Creating Loaders
test_dataset= TensorDataset(x,y,w)
test_dataloader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
for x,y,w in test_dataloader:
    print(x.shape)
    print(y.shape)
    print(w.shape)
    break

In [ ]:
predictions=[]
targets=[]
weights=[]
model.eval()
with torch.no_grad():
    for inputs,y,w in test_dataloader:
        # Get the model outputs (predictions) for the current batch
        inputs.to(device)
        outputs = model(inputs)  # inputs[0] since DataLoader gives a tuple (input, label)

        # Append the outputs to the predictions list
        predictions.append(outputs)
        targets.append(y)
        weights.append(w)

# Concatenate all predictions into a single tensor
test_predictions = torch.cat(predictions, dim=0)
test_targets = torch.cat(targets, dim=0)
test_weights = torch.cat(weights, dim=0)
print(test_predictions.shape)

cpu = torch.device("cpu")
test_predictions = test_predictions.to(cpu)
w_r2 = weighted_r2(test_predictions,test_targets,test_weights)
print(w_r2)
# find weighted R2 scores


In [ ]:
#Evaluation with online training

In [ ]:
model1=model
model1.eval()

In [ ]:
# prepare data
test_data = part_1.collect()
data = eval_prepare(test_data)
print(data.shape)


In [ ]:
# prepare data

data = torch.tensor(data)
data[:,4:83]= normalize(data[:,4:83])
data.shape


In [ ]:
data.shape

In [ ]:
data = torch.tensor(data)
data[:,4:83]= normalize(data[:,4:83])
data.shape
batches = batch_symbols(data)
print(len(batches))

In [ ]:

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)


In [ ]:
sequence = batches[1:6]

# Convert the selected sequence to a tensor if needed (e.g., if x_batches is a list)
sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
sequence_tensor= sequence_tensor[:,:,:]
sequence_tensor=sequence_tensor.permute(1,0,2)
sequence_tensor.shape

In [ ]:
sequence_tensor[10,:,0:3]

In [ ]:
predictions=[]
targets =[]
weights =[]
sequence_size = 5
for batch_idx, inputs in enumerate(batches[0:1600 - sequence_size + 1]):
    # Select sequences of size 5 with a sliding window
        sequence = batches[batch_idx:batch_idx + sequence_size]

    # Convert the selected sequence to a tensor if needed (e.g., if x_batches is a list)
        sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
        sequence_tensor= sequence_tensor[:,:,:]
        sequence_tensor=sequence_tensor.permute(1,0,2)
        inputs = sequence_tensor[:,:,4:83]
        y= sequence_tensor[:,-1,83]
        w= sequence_tensor[:,-1,3]

    # Feed sequence_tensor to the LSTM
        outputs = model1(inputs)
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass

        predictions.append(outputs.squeeze())  # Store predictions

        # Compute loss
        targets.append(y.squeeze())
        weights.append(w.squeeze())

        loss = w_mse(outputs.squeeze(), y.float(),w.float())

        # Backward pass
        loss.backward()
        optimizer.step()



print(f"Stored predictions: {len(predictions)} batches")

In [ ]:
weights[100].shape

In [ ]:

#prepare x,y,w for wr2
y=torch.cat([tensor for tensor in targets])
w=torch.cat([tensor for tensor in weights])
test_predictions=torch.cat([tensor for tensor in predictions])
print(test_predictions.shape)
print(y.shape)
print(w.shape)



In [ ]:
w[0]

In [ ]:

wr2 = weighted_r2(test_predictions,y,w)
wr2

In [ ]:
r2_scores["lstm no symbol no shuffle 1 layer"] = wr2.item()
print(r2_scores)


MODELS WITH SYMBOL EMBEDDINGS

In [ ]:
data = part_0.collect()

In [ ]:
sequences = prepare_sequences(data,5)
print(sequences.shape)


In [ ]:
# Create a dataset and dataloader
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dataset = EmbeddedDataset(sequences)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

In [ ]:
#Sanity check

model = ELSTM(39, 10, 79,256,1)
for inputs, targets, weights,symbols in dataloader:
    outputs = model(inputs,symbols)
    print(outputs.squeeze().shape)
    print(inputs.shape)
    print(targets.shape)
    print(weights.shape)
    print(symbols.shape)
    break

In [ ]:
#Train The model
model = ELSTM(39, 10, 79,256,1)
model = model.to(device)
model = torch.nn.DataParallel(model)
criterion = nn.MSELoss()

# Use Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model


In [ ]:
for epoch in range(10):  # Number of epochs
    for features,targets,weights,symbol_ids in dataloader:
        features = features.to(device)
        targets = targets.to(device)
        weights = weights.to(device)
        symbol_ids = symbol_ids.to(device)
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(features, symbol_ids)


        # Compute loss
        loss = criterion(predictions, targets)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

In [ ]:
#ONLINE EVALUATION

In [ ]:
model1.load_state_dict(torch.load(model_path))
model1.eval()  # Set model to evaluation mode

In [ ]:
# prepare data
test_data = part_1.collect()
data = eval_prepare(test_data)
print(data.shape)




In [ ]:
data = torch.tensor(data)
data[:,4:83]= normalize(data[:,4:83])
data.shape

In [ ]:

batches = batch_symbols(data)
print(len(batches))
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)

In [ ]:
sequence = batches[1:6]

# Convert the selected sequence to a tensor if needed
sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
sequence_tensor= sequence_tensor[:,:,:]
sequence_tensor=sequence_tensor.permute(1,0,2)
sequence_tensor.shape


In [ ]:
predictions=[]
targets =[]
weights =[]
sequence_size = 5
for batch_idx, inputs in enumerate(batches[0:1600 - sequence_size + 1]):
    # Select sequences of size 5 with a sliding window
        sequence = batches[batch_idx:batch_idx + sequence_size]

    # Convert the selected sequence to a tensor if needed (e.g., if x_batches is a list)
        sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
        sequence_tensor= sequence_tensor[:,:,:]
        sequence_tensor=sequence_tensor.permute(1,0,2)
        inputs = sequence_tensor[:,:,4:83]
        y= sequence_tensor[:,-1,83]
        w= sequence_tensor[:,-1,3]
        symbols = sequence_tensor[:,:,2].to(int)

    # Feed sequence_tensor to the LSTM
        outputs = model1(inputs,symbols)
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass

        predictions.append(outputs.squeeze())  # Store predictions

        # Compute loss
        targets.append(y.squeeze())
        weights.append(w.squeeze())

        loss = w_mse(outputs.squeeze(), y.float(),w.float())

        # Backward pass
        loss.backward()
        optimizer.step()



print(f"Stored predictions: {len(predictions)} batches")

In [ ]:
y=torch.cat([tensor for tensor in targets])
w=torch.cat([tensor for tensor in weights])
test_predictions=torch.cat([tensor for tensor in predictions])
print(test_predictions.shape)
print(y.shape)
print(w.shape)
wr2 = weighted_r2(test_predictions,y,w)
wr2

In [ ]:

r2_scores["embedded lstm 1 layer no shuffle"] = wr2.item()
print(r2_scores)

In [ ]:
# Memory management if needed
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Mixture of Experts

In [ ]:
part0 = part_0.collect()
data = expert_data(part0)
print(len(data))
print(data[0].shape)

######

In [ ]:
def expert_sequences(tensor_data, seq_length):
    # Sort the data based on the first and second column
    sorted_data = tensor_data[torch.argsort(tensor_data[:, 0])]  # Sort by first column
    sorted_data = sorted_data[torch.argsort(sorted_data[:, 1])]  # Then sort by second column

    # Create sequences using sliding window
    sequences = []
    for i in range(len(sorted_data) - seq_length + 1):
        seq = sorted_data[i:i + seq_length]
        sequences.append(seq)

    return torch.stack(sequences)

In [ ]:
data_repo={}
for symbol_data in data:
    symbol_id = int(symbol_data[0,2].to(int))
    data_repo[symbol_id] = expert_sequences(symbol_data,5)


######

In [ ]:
data_repo[0].shape

In [ ]:

# Create subsets for each symbol
symbol_datasets = {symbol: FullDataset(data) for symbol, data in data_repo.items()}

dataloaders = {symbol: DataLoader(dataset, batch_size=256, shuffle=True) for symbol, dataset in symbol_datasets.items()}

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
models = {}
criterion = nn.MSELoss()
for symbol, dataloader in dataloaders.items():
        model = LSTMModel(input_size=79, hidden_size=256, output_size=1)
        model.to(device)
        model = torch.nn.DataParallel(model)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
        for epoch in range(10):
            for features,targets,weights,symbols in dataloader:
                features= features.to(device)
                targets= targets.to(device)
                weights= weights.to(device)
                optimizer.zero_grad()
                predictions = model(features)
                loss = criterion(predictions.squeeze(), targets.squeeze())
                loss.backward()
                optimizer.step()

        # Store trained model
        models[symbol] = model


In [ ]:
models[0]

In [ ]:
#Online training evaluation
for k,v in sorted(models.items()):
    print(k)

In [ ]:
test_data = part_1.collect()
data = eval_prepare(test_data)
data.shape



In [ ]:
data = torch.tensor(data)
data[:,4:83] = normalize(data[:,4:83])
batches = batch_symbols(data)
loss_fn = nn.MSELoss()

In [ ]:
sequence = batches[1:6]

# Convert the selected sequence to a tensor if needed (e.g., if x_batches is a list)
sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
sequence_tensor= sequence_tensor[:,:,:]
sequence_tensor=sequence_tensor.permute(1,0,2)
sequence_tensor.shape


In [ ]:
sequence_tensor[0,:,2][0].to(int).item()

In [ ]:
input = sequence_tensor[0].unsqueeze(0)

In [ ]:
input[:,0,2].to(int).item()

In [ ]:
optimizers={}
for key,model in models.items():
    optimizers[key] = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
predictions=[]
targets =[]
weights =[]
sequence_size = 5
for batch_idx, inputs in enumerate(batches[0:1600 - sequence_size + 1]):
# Select sequences of size 5 with a sliding window
    sequence = batches[batch_idx:batch_idx + sequence_size]

# Convert the selected sequence to a tensor if needed (e.g., if x_batches is a list)
    sequence_tensor = torch.stack(sequence)  # Shape: (sequence_size, input_size)
    sequence_tensor= sequence_tensor[:,:,:]
    sequence_tensor=sequence_tensor.permute(1,0,2)

    batch_features = []
    batch_targets = []
    batch_weights = []
    batch_symbols = []

    for i, input in enumerate(sequence_tensor):
        input = input.unsqueeze(0)
        features = input[:,:,4:83]
        y= input[:,-1,83]
        w= input[:,-1,3]
        symbol = input[:,0,2].to(int).item()


        # Check if symbol is in models and optimizers
        if symbol not in models or symbol not in optimizers:
            continue

        batch_features.append(features)
        batch_targets.append(y)
        batch_weights.append(w)
        batch_symbols.append(symbol)

    # Check if batch is not empty
    if not batch_features:
        continue

    # Stack batch features, targets, and weights
    batch_features = torch.stack(batch_features)
    batch_targets = torch.stack(batch_targets)
    batch_weights = torch.stack(batch_weights)

    # Zero gradients
    for symbol in set(batch_symbols):  # Use set to avoid redundant `zero_grad` calls
        optimizers[symbol].zero_grad()

    # Forward pass
    batch_outputs = []
    for symbol, features in zip(batch_symbols, batch_features):
        output = models[symbol](features)
        batch_outputs.append(output)

    # Compute loss
    batch_loss = 0
    for output, target, weight in zip(batch_outputs, batch_targets, batch_weights):
        output = output.to(device)
        target = target.to(device)
        weight =weight.to(device)
        loss = w_mse(output.squeeze(), target.float(), weight.float())
        batch_loss += loss

    # Backward pass
    batch_loss.backward()
    for symbol in set(batch_symbols):
        optimizers[symbol].step()

    # Store predictions, targets, and weights
    predictions.extend(batch_outputs)
    targets.extend(batch_targets)
    weights.extend(batch_weights)

print(f"Stored predictions: {len(predictions)}")
print(f"Stored targets: {len(targets)}")
print(f"Stored weights: {len(weights)}")

In [ ]:
predictions = [el.detach() for el in predictions]
weights = [el.detach() for el in weights]
targets = [el.detach() for el in targets]

In [ ]:
predictions = torch.tensor(predictions)
weights =  torch.tensor(weights)
targets = torch.tensor(targets)

In [ ]:
print(predictions.shape)
print(weights.shape)
print(targets.shape)


In [ ]:
# NOTEBOOK SUBMISSION
wr2 = weighted_r2(predictions,targets,weights)
print(wr2)

In [ ]:
r2_scores["lstm 1 layer experts"] = wr2.item()
print(r2_scores)

In [ ]:
#XGboost
import xgboost as xgb

In [ ]:
data = part_0.collect()
cols = data.columns[4:83]
data= data.fill_null(0)
epsilon = 1e-6  # A small threshold to avoid division by near-zero values

# Calculate standard deviations for the columns before applying normalization
std_devs = {col: data[col].std() for col in cols}

# Now apply standardization while checking the condition
standardized_data = data.with_columns(
    [
        ((pl.col(col) - pl.col(col).mean()) / (std_devs[col] + epsilon)).alias(col)
        for col in cols if std_devs[col] > epsilon
    ]
)

In [ ]:
for col in standardized_data.columns:
    mean = standardized_data[col].mean()
    std = standardized_data[col].std()
    print(f"{col}: mean={mean}, std={std}")


In [ ]:
cols_to_remove = [
"date_id",
"time_id",
"weight",
'responder_0',
'responder_1',
'responder_2',
'responder_3',
'responder_4',
'responder_5',
'responder_7',
'responder_8'
]

# Drop the specified columns
X = data.drop(cols_to_remove)
y=X["responder_6"]
X=X.drop("responder_6")

In [ ]:
feature_types = ['c' if col == 'symbol_id' else 'float' for col in X.columns]
dmatrix = xgb.DMatrix(data=X, label=y, feature_types=feature_types)


In [ ]:
params = {
    "objective": "reg:squarederror",  # Standard regression objective
    "max_depth": 6,                  # Maximum tree depth
    "eta": 0.01,                      # Learning rate
    "eval_metric": "rmse"            # Metric for evaluation (Root Mean Squared Error)
}


In [ ]:
# Train the model
num_round = 100  # Number of boosting rounds
bst = xgb.train(params=params, dtrain=dmatrix, num_boost_round=100)


In [ ]:
test_data = part_1.collect()
cols = test_data.columns[4:83]
data= test_data.fill_null(0)
epsilon = 1e-6  # A small threshold to avoid division by near-zero values

# Calculate standard deviations for the columns before applying normalization
std_devs = {col: data[col].std() for col in cols}

# Now apply standardization while checking the condition
test_data = test_data.with_columns(
    [
        ((pl.col(col) - pl.col(col).mean()) / (std_devs[col] + epsilon)).alias(col)
        for col in cols if std_devs[col] > epsilon
    ]
)

cols_to_remove = [
"date_id",
"time_id",
'responder_0',
'responder_1',
'responder_2',
'responder_3',
'responder_4',
'responder_5',
'responder_7',
'responder_8'
]

# Drop the specified columns
X_test = test_data.drop(cols_to_remove)
y_test=X_test["responder_6"]
X_test=X_test.drop("responder_6")
w_test=X_test["weight"]
X_test = X_test.drop("weight")

test_dmatrix = xgb.DMatrix(data=X_test)
predictions = bst.predict(test_dmatrix)


In [ ]:
predictions = torch.tensor(predictions)
y_test = torch.tensor(y_test)
w_test = torch.tensor(w_test)

In [ ]:
wr2 = weighted_r2(predictions,y_test,w_test)
print(wr2)

In [ ]:
r2_scores["XGboost lr =0.01 with no online training"] = wr2.item()
print(r2_scores)

In [ ]:
# evaluation of XGBOOST with online training
data = part_1.collect()
data = eval_prepare(data)
data = torch.tensor(data)
batches = batch_symbols(data)

In [ ]:
predictions = []
targets = []
weights = []

for batch_idx, inputs in enumerate(batches[0:10000]):
    w = inputs[:, 3]  # Extracting weights (ensure column index is correct)



    # Preparing the input tensor by excluding the 4th column
    inputs = torch.cat([inputs[:, :3], inputs[:, 4:]], dim=1)  # Corrected slice



    # Ensure inputs are properly converted (if necessary)
    inputs[:, 2] = inputs[:, 2].to(int)


    X_tens = inputs[:, 2:82]  # Select features for the model
    y_test = inputs[:, 82]  # Select target values

    # Prepare DMatrix for XGBoost
    dmatrix = xgb.DMatrix(data=X_tens, label=y_test)

    # Online training (updating the model)

    # Get model predictions for the current batch
    outputs = bst.predict(dmatrix)
    predictions.append(outputs.squeeze())  # Store predictions
    bst.update(dmatrix, 100)
    targets.append(y_test.squeeze())  # Store targets
    weights.append(w.squeeze())  # Store weights

print(f"Stored predictions: {len(predictions)} batches")


In [ ]:
y=torch.cat([tensor for tensor in targets])
w=torch.cat([tensor for tensor in weights])
test_predictions = torch.cat([torch.tensor(tensor) for tensor in predictions])
print(test_predictions.shape)
print(y.shape)
print(w.shape)
wr2 = weighted_r2(test_predictions,y,w)
wr2

In [ ]:
wr2 = weighted_r2(predictions,y_test,w_test)
print(wr2)